# HP lattice protein folding

*biophysics · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/biophysics/hp_lattice_folding)

The simplest protein model that still folds: a chain of hydrophobic (H) and polar (P) residues on a square lattice, where every H–H contact between residues that are not chain neighbours lowers the energy by one. Many replicas of a 20-residue benchmark fold at once by Metropolis Monte Carlo with pull and crankshaft moves while the temperature is lowered, and the contact map comes out of one pairwise-distance tensor of shape $(R, N, N)$.

**Vector form**

$$E(\{\mathbf r\}) = -\varepsilon \sum_{i < j - 1} h_i\, h_j\, \big[\,\lVert \mathbf r_i - \mathbf r_j \rVert = 1\,\big],\qquad \lVert \mathbf r_{i+1} - \mathbf r_i \rVert = 1,\quad \mathbf r_i \ne \mathbf r_j,\qquad P_{\mathrm{acc}} = \min\!\big(1, e^{-\Delta E / T}\big)$$

**Index (tensor) form**

$$d_{rij} = x_{rik}x_{rik} + x_{rjk}x_{rjk} - 2\,x_{rik}x_{rjk} \qquad C_{rij} = [d_{rij} = 1]\,[|i - j| > 1] \qquad E_r = -\tfrac12\, h_i\, C_{rij}\, h_j$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{rik}$ | `straight_chains(R, N, device)` | (R, N, 2) | lattice coordinate $k$ of residue $i$ in replica $r$; every replica starts as a straight rod |
| $h_i$ | `hydrophobic(seq, device)` | (N,) | 1 for H, 0 for P |
| $x_{rik}x_{rjk}$ | `einsum('rik,rjk->rij', x, x)` | (R, N, N) | repeated $k$ ⇒ summed: a batched Gram matrix; with the squared norms it gives $d_{rij} = \lVert \mathbf r_i - \mathbf r_j\rVert^2$ |
| $C_{rij}$ | `contact_map(d)` | (R, N, N) | lattice neighbours that are not chain neighbours |
| $h_i C_{rij} h_j$ | `einsum('i,rij,j->r', h, C, h)` | (R,) | repeated $i$ and $j$ ⇒ summed: H–H contacts, each counted twice, hence the $\tfrac12$ |
| $d_{rij} \ne 0$ | `self_avoiding(x, d)` | (R,) | bonds of length 1 and no two residues on one site, otherwise the move is rejected |
| $T_s$ | `temperatures(T_start, T_end, S, device)` | (S,) | annealing schedule: $T$ falls geometrically, step by step |
| $P_{\mathrm{acc}}$ | `u < exp(-dE / T)` | (R,) | Metropolis rule: downhill always, uphill with probability $e^{-\Delta E/T}$ |

Proteins fold largely because hydrophobic side chains hide from water in a compact core. The HP model (Dill, 1985) keeps only that: an H–H contact is worth $-\varepsilon$, nothing else counts, and the chain is a self-avoiding walk on the lattice. Even so, finding the lowest energy is NP-hard, and the number of conformations grows like $2.64^N$. The default sequence HPHPPHHPHPPHPHHPPHPH is the classic 20-residue benchmark of Unger and Moult (1993) with ground state $E = -9$; the others have known minima of $-9$, $-8$ and $-14$. Each step, every replica picks one residue and tries a pull move (Lesh, Mitzenmacher and Whitesides, 2003: the residue jumps to a free corner and drags its neighbours along the chain until the chain is connected again; at the ends this rotates the last residue, and in the simplest case it is a corner flip) or a crankshaft move (a U-shaped pair flips over). A proposal that breaks a bond or puts two residues on one site is rejected; otherwise the Metropolis rule decides. High temperature lets the chains explore, slow cooling lets them settle into compact folds with an H core and P residues on the surface. The energy $E$ is updated by $\Delta E$ of each accepted move and checked against a full recount in the tests. Pull moves are not exactly symmetric proposals, so the run is an optimiser rather than an exact thermal sampler. Try the 36-residue sequence: it needs more steps or replicas, and the share of replicas that reach the ground state shows how rugged its energy landscape is.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/biophysics/hp_lattice_folding/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import contact_map, squared_distances


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    seq, e_ground = tensors.sequence(params["sequence"])
    h = tensors.hydrophobic(seq, dev)  # (N,)
    R = params["replicas"]
    n_steps = 300 if fast_mode() else params["steps"]
    x0 = tensors.straight_chains(R, len(seq), dev)  # (R, N, 2)
    temps = tensors.temperatures(params["T_start"], params["T_end"], n_steps, dev)  # (S,)
    print(f"sequence {seq} (N = {len(seq)}, {int(h.sum())} H), known ground state E = {e_ground}")
    print(f"{R} replicas × {n_steps} Monte Carlo steps, T from {params['T_start']} to {params['T_end']}")

    out = solve.anneal(x0, h, temps, n_records=60, seed=0)
    E, best_E = out["E"], out["best_E"]
    print(f"acceptance {out['acceptance']:.1%}")
    for k in (0, len(out["record_steps"]) // 4, len(out["record_steps"]) // 2, len(out["record_steps"]) - 1):
        e, s = out["energies"][k], out["record_steps"][k].item()
        print(f"step {s:5d}: mean E = {e.mean().item():6.2f}   lowest E = {int(e.min().item()):3d}")
    found = (best_E == e_ground).float().mean().item()
    print(
        f"lowest energy reached: {best_E.min().item():.0f} (ground state {e_ground}); {found:.0%} of replicas found it"
    )

    r = int(E.argmin())  # the replica that ends lowest is the one we watch fold
    C = contact_map(squared_distances(out["x"][r : r + 1]))[0]
    hh = [(i, j) for i in range(len(seq)) for j in range(i + 2, len(seq)) if C[i, j] and seq[i] == seq[j] == "H"]
    print(f"replica {r}: E = {E[r].item():.0f}, H–H contacts {hh}")
    return plot.animate(out, r, seq, e_ground, out_dir / "hp_lattice_folding.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/biophysics/hp_lattice_folding/equation.py
r"""Physics → tensor translation for the HP lattice protein.

A protein is a chain of N residues on the square lattice, x_{rik} = coordinate k of residue i in replica r.
Residues are either hydrophobic (h_i = 1) or polar (h_i = 0). Two residues that touch on the lattice but are not
neighbours along the chain form a contact, and every H–H contact lowers the energy by ε = 1:

    d_{rij} = (x_{rik} - x_{rjk}) (x_{rik} - x_{rjk})        summed over k:   squared distance
    C_{rij} = [d_{rij} = 1] [|i - j| > 1]                     contact map
    E_r     = -½ h_i C_{rij} h_j                              summed over i and j (½: each pair counted twice)

A chain is allowed only if consecutive residues are one lattice step apart and no two residues share a site
(d_{rij} > 0 for i ≠ j): a self-avoiding walk.
"""

import torch


def squared_distances(x: torch.Tensor) -> torch.Tensor:
    """d_{rij} = |x_ri|² + |x_rj|² - 2 x_{rik} x_{rjk}, repeated k summed.   x: (R, N, 2) → (R, N, N).

    This is (x_ri - x_rj)·(x_ri - x_rj) expanded, so the pair term is one batched matrix product.
    """
    x = x - x[:, :1]  # put residue 0 at the origin: distances don't change and the numbers stay small
    norm = torch.einsum("rik,rik->ri", x, x)  # (R, N)
    gram = torch.einsum("rik,rjk->rij", x, x)  # (R, N, N)
    return norm[:, :, None] + norm[:, None, :] - 2 * gram


def contact_map(d: torch.Tensor) -> torch.Tensor:
    """C_{rij} = [d_{rij} = 1] [|i - j| > 1]: lattice neighbours that are not chain neighbours.   (R, N, N)."""
    n = d.shape[1]
    idx = torch.arange(n, device=d.device)
    non_bonded = (idx[:, None] - idx[None, :]).abs() > 1  # (N, N)
    return ((d == 1) & non_bonded).float()


def energy(C: torch.Tensor, h: torch.Tensor) -> torch.Tensor:
    """E_r = -½ h_i C_{rij} h_j, repeated i and j summed.   C: (R, N, N), h: (N,) → (R,)."""
    return -0.5 * torch.einsum("i,rij,j->r", h, C, h)


def chain_energy(x: torch.Tensor, h: torch.Tensor) -> torch.Tensor:
    """Energy straight from the coordinates: x → d → C → E.   (R,)."""
    return energy(contact_map(squared_distances(x)), h)


def self_avoiding(x: torch.Tensor, d: torch.Tensor) -> torch.Tensor:
    """True for replicas whose bonds all have length 1 and whose residues all sit on different sites.   (R,) bool.

    x: (R, N, 2) coordinates, d: (R, N, N) their squared distances.
    """
    n = x.shape[1]
    bonds = (x[:, 1:] - x[:, :-1]).abs().sum(-1)  # (R, N-1) lattice step lengths
    overlap = (d == 0) & ~torch.eye(n, dtype=torch.bool, device=x.device)
    return (bonds == 1).all(-1) & ~overlap.any(-1).any(-1)


def metropolis(dE: torch.Tensor, T: float, u: torch.Tensor) -> torch.Tensor:
    """Accept a move with probability min(1, exp(-ΔE / T)).   dE, u: (R,) with u uniform in [0, 1)."""
    return u < torch.exp(-dE / T)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/biophysics/hp_lattice_folding/tensors.py
"""Build the concrete tensors: the H/P sequence, the starting chains of every replica and the cooling schedule.

Lattice positions are integer (x, y) pairs, so a chain of N residues in R replicas is a (R, N, 2) tensor. It is
stored as float (whole numbers are exact in float32) so distances can use fast matrix products.
"""

import math

import torch

# Standard 2D HP benchmarks (Unger & Moult 1993) with their known ground-state energies.
SEQUENCES = {
    "benchmark_20": ("HPHPPHHPHPPHPHHPPHPH", -9),
    "benchmark_24": ("HHPPHPPHPPHPPHPPHPPHPPHH", -9),
    "benchmark_25": ("PPHPPHHPPPPHHPPPPHHPPPPHH", -8),
    "benchmark_36": ("PPPHHPPHHPPPPPHHHHHHHPPHHPPPPHHPPHPP", -14),
}


def sequence(name: str) -> tuple[str, int]:
    """The H/P string and its known lowest energy."""
    return SEQUENCES[name]


def hydrophobic(seq: str, device: torch.device) -> torch.Tensor:
    """h_i = 1 for H (hydrophobic) and 0 for P (polar): shape (N,)."""
    return torch.tensor([1.0 if c == "H" else 0.0 for c in seq], device=device)


def straight_chains(n_replicas: int, n: int, device: torch.device) -> torch.Tensor:
    """x_{rik}: every replica starts fully stretched along x, residue i at (i, 0): shape (R, N, 2)."""
    line = torch.stack([torch.arange(n, dtype=torch.float32), torch.zeros(n)], dim=-1)  # (N, 2)
    return line.expand(n_replicas, n, 2).clone().to(device)


def temperatures(t_start: float, t_end: float, n_steps: int, device: torch.device) -> torch.Tensor:
    """Simulated annealing: T_s falls geometrically from t_start to t_end (units of the contact energy ε/k_B): (S,)."""
    return torch.logspace(math.log10(t_start), math.log10(t_end), n_steps, device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/biophysics/hp_lattice_folding/solve.py
"""Metropolis Monte Carlo with pull and crankshaft moves, run for all replicas at once, while the temperature falls.

Every step, each replica proposes one move. Moves that break the chain or make it cross itself are rejected; the
rest are accepted with the Metropolis rule. All R replicas advance together as one (R, N, 2) tensor.
"""

import torch

from problems._common.progress import steps

from .equation import contact_map, energy, metropolis, self_avoiding, squared_distances


def pull_move(x: torch.Tensor, i: torch.Tensor, side: torch.Tensor) -> torch.Tensor:
    """Pull move (Lesh et al. 2003) towards residue 0.   x: (R, N, 2), i: (R,) in [0, N-2], side: (R,) = ±1.

    Residue i jumps to L, a free corner next to residue i+1; residue i-1 follows to C, the corner next to both L and
    the old site of i; then every residue j < i-1 moves two places up the chain (to the old site of j+2) until one
    already touches the new site of j+1. With C equal to the old site of i-1 this is a simple corner flip, and with
    i = 0 it rotates the end residue.
    """
    r = torch.arange(x.shape[0], device=x.device)
    j = torch.arange(x.shape[1], device=x.device)[None, :]  # (1, N)
    i_ = i[:, None]  # (R, 1)
    bond = x[r, i + 1] - x[r, i]  # (R, 2) unit lattice step from i to i+1
    perp = side[:, None] * torch.stack([-bond[:, 1], bond[:, 0]], dim=-1)  # (R, 2) one of the two sideways steps
    L = x[r, i + 1] + perp  # new site of residue i
    C = x[r, i] + perp  # new site of residue i-1

    candidate = torch.where((j <= i_ - 2)[..., None], torch.roll(x, -2, dims=1), x)  # residue j → old site of j+2
    candidate = torch.where((j == i_ - 1)[..., None], C[:, None], candidate)
    candidate = torch.where((j == i_)[..., None], L[:, None], candidate)

    # residue j < i stays where it is if it already touches the new site of j+1 — and then so do all below it
    touches = (x - torch.roll(candidate, -1, dims=1)).abs().sum(-1) == 1  # (R, N)
    keeps_going = (~touches | (j >= i_)).int()
    moves = keeps_going.flip(1).cumprod(1).flip(1).bool() & (j <= i_)  # (R, N)
    return torch.where(moves[..., None], candidate, x)


def crankshaft_move(x: torch.Tensor, i: torch.Tensor) -> torch.Tensor:
    """Flip the U-shaped pair (i, i+1) to the other side of the line from i-1 to i+2.   i: (R,) in [1, N-3].

    x'_i = 2 x_{i-1} - x_i,   x'_{i+1} = 2 x_{i+2} - x_{i+1}.   For any shape other than a U the bonds break,
    and self_avoiding() rejects the move.
    """
    r = torch.arange(x.shape[0], device=x.device)
    new = x.clone()
    new[r, i] = 2 * x[r, i - 1] - x[r, i]
    new[r, i + 1] = 2 * x[r, i + 2] - x[r, i + 1]
    return new


def propose(x: torch.Tensor, u: torch.Tensor) -> torch.Tensor:
    """One random move per replica: a pull from either end of the chain (80 %) or a crankshaft (20 %).

    u: (R, 4) uniform random numbers that pick the residue, the side, the chain end and the move type.
    """
    R, N, _ = x.shape
    i = (u[:, 0] * (N - 1)).long()  # residue in [0, N-2]
    side = torch.where(u[:, 1] < 0.5, -1.0, 1.0)
    from_end = u[:, 2] < 0.5
    crank = u[:, 3] < 0.2

    # pulling towards the far end = the same pull on the chain read backwards
    flipped = torch.where(from_end[:, None, None], x.flip(1), x)
    pulled = pull_move(flipped, i, side)
    pulled = torch.where(from_end[:, None, None], pulled.flip(1), pulled)

    cranked = crankshaft_move(x, i.clamp(1, N - 3))
    return torch.where(crank[:, None, None], cranked, pulled)


def anneal(x0: torch.Tensor, h: torch.Tensor, temps: torch.Tensor, n_records: int, seed: int = 0) -> dict:
    """Run len(temps) Monte Carlo steps. Returns snapshots of every replica and the energy bookkeeping.

    E is never recomputed from scratch inside the loop: it is updated by ΔE of each accepted move, so the tests can
    check that it still matches a fresh recount at the end.
    """
    gen = torch.Generator().manual_seed(seed)
    n_steps = len(temps)
    every = max(1, n_steps // n_records)
    x = x0.clone()
    E = energy(contact_map(squared_distances(x)), h)  # (R,)
    best_E, best_x = E.clone(), x.clone()
    snapshots, energies, record_steps = [x.clone()], [E.clone()], [0]
    accepted = torch.zeros((), device=x.device)

    for s, T in zip(steps(n_steps, "MC steps"), temps.tolist()):
        u = torch.rand(x.shape[0], 5, generator=gen).to(x.device)  # all random numbers of this step
        new = propose(x, u[:, :4])
        d = squared_distances(new)  # (R, N, N)
        dE = energy(contact_map(d), h) - E
        accept = self_avoiding(new, d) & metropolis(dE, T, u[:, 4])  # (R,)
        x = torch.where(accept[:, None, None], new, x)
        E = E + torch.where(accept, dE, torch.zeros_like(dE))
        accepted += accept.sum()

        better = E < best_E
        best_E = torch.where(better, E, best_E)
        best_x = torch.where(better[:, None, None], x, best_x)
        if (s + 1) % every == 0 or s == n_steps - 1:
            snapshots.append(x.clone())
            energies.append(E.clone())
            record_steps.append(s + 1)

    return {
        "x": x,  # (R, N, 2) final chains
        "E": E,  # (R,) energy kept by ΔE bookkeeping
        "best_x": best_x,  # (R, N, 2) lowest-energy chain each replica ever visited
        "best_E": best_E,  # (R,)
        "snapshots": torch.stack(snapshots),  # (K, R, N, 2)
        "energies": torch.stack(energies),  # (K, R)
        "record_steps": torch.tensor(record_steps),  # (K,)
        "acceptance": accepted.item() / (n_steps * x.shape[0]),
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/biophysics/hp_lattice_folding/plot.py
"""Animate the chain that ends lowest as it folds, the energy against Monte Carlo step, and its contact map."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

from problems._common.plotting import frame_indices, save_gif

from .equation import contact_map, squared_distances

# contact map colours: nothing, chain bond, contact with a P residue, H–H contact
CMAP = ListedColormap(["#ffffff", "#e5e7eb", "#93c5fd", "#1e40af"])


def animate(out: dict, r: int, seq: str, e_ground: int, path: Path) -> Path:
    snaps = out["snapshots"][:, r]  # (K, N, 2) replica r at every recorded step
    maps = _contact_levels(snaps, seq)  # (K, N, N)
    chains = snaps.cpu().numpy()
    chains = chains - np.round(chains.mean(axis=1, keepdims=True))  # keep the chain in view
    energies = out["energies"].cpu().numpy()  # (K, R)
    step = out["record_steps"].numpy()
    is_h = np.array([c == "H" for c in seq])
    n = len(seq)

    fig, (ax_c, ax_e, ax_m) = plt.subplots(1, 3, figsize=(10, 3.7), gridspec_kw={"width_ratios": [1, 1.25, 1]})

    # --- the chain on the lattice ---
    half = np.maximum(np.abs(chains).max(axis=(1, 2)) + 1, 4)  # (K,) zoom on the chain in every frame
    ax_c.set_aspect("equal")
    ax_c.set_xticks([])
    ax_c.set_yticks([])
    ax_c.grid(False)
    for g in np.arange(-n, n + 1):  # the square lattice, drawn once
        ax_c.axhline(g, color="#e5e7eb", lw=0.6, zorder=0)
        ax_c.axvline(g, color="#e5e7eb", lw=0.6, zorder=0)
    (contacts,) = ax_c.plot([], [], "--", color="#dc2626", lw=1.5)
    (backbone,) = ax_c.plot([], [], "-", color="#374151", lw=2)
    (h_dots,) = ax_c.plot([], [], "o", ms=7, color="#2563eb", mec="#1e40af", label="H")
    (p_dots,) = ax_c.plot([], [], "o", ms=7, color="white", mec="#374151", label="P")
    ax_c.legend(loc="upper right", fontsize=7, handletextpad=0.2, borderpad=0.3)
    title = ax_c.set_title("step 0, E = 0", fontsize=10)  # real text so tight_layout leaves room

    # --- energy against step ---
    ax_e.plot(step, energies.mean(axis=1), color="#9ca3af", lw=1.2, label="mean of replicas")
    ax_e.plot(step, energies.min(axis=1), color="#2563eb", lw=1.2, label="lowest replica")
    ax_e.plot(step, energies[:, r], color="#ea580c", lw=1.2, label="replica shown")
    ax_e.axhline(e_ground, color="#16a34a", ls="--", lw=1, label=f"ground state {e_ground}")
    (marker,) = ax_e.plot([], [], "o", color="#ea580c", ms=5)
    ax_e.set_ylim(e_ground - 1, 0.5)
    ax_e.set_xlabel("Monte Carlo step (T falls →)")
    ax_e.set_ylabel(r"energy $E$ (units of $\varepsilon$)")
    ax_e.legend(fontsize=7, loc="upper right")
    ax_e.set_title("simulated annealing", fontsize=10)

    # --- contact map C_ij ---
    image = ax_m.imshow(maps[0], cmap=CMAP, vmin=0, vmax=3, interpolation="nearest")
    ticks = [i for i in range(0, n, 5)]
    ax_m.set_xticks(ticks)
    ax_m.set_yticks(ticks)
    ax_m.set_xlabel("residue $j$")
    ax_m.set_ylabel("residue $i$", labelpad=1)
    ax_m.grid(False)
    ax_m.set_title("contact map (dark: H–H)", fontsize=10)

    def update(k: int):
        c = chains[k]
        ax_c.set_xlim(-half[k] - 0.5, half[k] + 0.5)
        ax_c.set_ylim(-half[k] - 0.5, half[k] + 0.5)
        size = float(np.clip(50 / half[k], 3, 10))
        h_dots.set_markersize(size)
        p_dots.set_markersize(size)
        backbone.set_data(c[:, 0], c[:, 1])
        h_dots.set_data(c[is_h, 0], c[is_h, 1])
        p_dots.set_data(c[~is_h, 0], c[~is_h, 1])
        i, j = np.nonzero(np.triu(maps[k] == 3))
        xs = np.stack([c[i, 0], c[j, 0], np.full(len(i), np.nan)], axis=1).ravel()
        ys = np.stack([c[i, 1], c[j, 1], np.full(len(i), np.nan)], axis=1).ravel()
        contacts.set_data(xs, ys)
        title.set_text(f"step {step[k]}, E = {energies[k, r]:.0f}")
        marker.set_data([step[k]], [energies[k, r]])
        image.set_data(maps[k])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(step), 60), path, fps=8)


def _contact_levels(snaps, seq: str) -> np.ndarray:
    """Per snapshot an (N, N) map: 1 chain bond, 2 contact with a P residue, 3 H–H contact."""
    C = contact_map(squared_distances(snaps)).cpu().numpy()  # (K, N, N), the snapshots act as a batch
    is_h = np.array([c == "H" for c in seq])
    n = len(seq)
    idx = np.arange(n)
    levels = np.where(np.abs(idx[:, None] - idx[None, :]) == 1, 1, 0)[None].repeat(len(C), 0)
    levels = np.where(C == 1, 2, levels)
    return np.where((C == 1) & (is_h[:, None] & is_h[None, :]), 3, levels)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'sequence': 'benchmark_20',
    'replicas': 128,
    'steps': 5000,
    'T_start': 1.0,
    'T_end': 0.15
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.biophysics.hp_lattice_folding")]:
    del sys.modules[name]
main = importlib.import_module("problems.biophysics.hp_lattice_folding.main")
problem_dir = Path("problems/biophysics/hp_lattice_folding")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
